<a href="https://colab.research.google.com/github/vikobaldigi/DATA-71200-Advanced-Data-Analysis-Methods-Project/blob/main/Project%201/KovacevicVi_Project1_10_31.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load Dataset

In [2]:
# Load NYC Schools Dataset
df = pd.read_csv('/Users/vikobal/Regents_Data/machinelearning/schools_curated.csv')

print(f"Dataset Shape: {df.shape}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/vikobal/Regents_Data/machinelearning/schools_curated.csv'

# Train-Test Split with Stratification

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(['Subject', 'DBN', 'School_Name', 'School Type', 'School Level', 'Neighborhood'], axis=1)
y = df['Subject']

# Stratified train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTraining set class distribution:")
print(y_train.value_counts())
print(f"\nTest set class distribution:")
print(y_test.value_counts())

# Explore Training Set

In [ ]:
# Create training dataframe for exploration
train_set = X_train.copy()
train_set['Subject'] = y_train.values

print(f"Training set shape: {train_set.shape}")
train_set.head()

In [ ]:
# Dataset information
train_set.info()

In [ ]:
# Statistical summary
train_set.describe()

# Data Cleaning

In [ ]:
# Missing values in training set
missing_data = train_set.isnull().sum()
missing_percent = (missing_data / len(train_set)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_data,
    'Percentage': missing_percent
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print("Missing Values in Training Set (Before Cleaning):")
print("="*70)
print(missing_df)
print("\nNote: Columns with >30% missing will be dropped to avoid artificial data.")

In [ ]:
# Drop columns with >30% missing values
# These columns have too many missing values to impute
# We also have alternative demographic features (Asian_Percent, Black_Percent, etc.) that are complete

cols_to_drop = missing_df[missing_df['Percentage'] > 30].index.tolist()

print("Dropping columns with >30% missing data:")
print("="*70)
for col in cols_to_drop:
    pct = missing_df.loc[col, 'Percentage']
    print(f"  - {col}: {pct:.1f}% missing")

train_set = train_set.drop(columns=cols_to_drop)

print(f"\nFeatures after dropping high-missing columns: {len(train_set.columns)}")

In [ ]:
# Apply imputation to remaining missing values (~4% in teacher features)
from sklearn.impute import SimpleImputer

# Separate numerical columns
numerical_cols = train_set.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Create imputer and fit on training set
imputer = SimpleImputer(strategy='median')
imputer.fit(train_set[numerical_cols])

# Transform training set
train_set[numerical_cols] = imputer.transform(train_set[numerical_cols])

print(f"\nMissing values after imputation: {train_set.isnull().sum().sum()}")
print(f"Final feature count: {len(train_set.columns)}")
print("\n✓ Data cleaning complete! All remaining missing values handled with median imputation.")

In [ ]:
# Apply same cleaning to test set
test_set = X_test.copy()
test_set['Subject'] = y_test.values

# Drop same columns
test_set = test_set.drop(columns=cols_to_drop)

# Apply imputation
test_set[numerical_cols] = imputer.transform(test_set[numerical_cols])

print(f"Test set missing values after cleaning: {test_set.isnull().sum().sum()}")
print(f"Test set feature count: {len(test_set.columns)}")
print("\n✓ Test set cleaned successfully!")

In [ ]:
# Histogram of all numerical features in training set
train_set.hist(bins=30, figsize=(20, 15))
plt.suptitle('Distribution of All Features in Training Set', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter matrix of selected features
scatter_features = ['MeanScore', 'Teacher_Count', 'Total_Staff', 'Hispanic_Percent', 'White_Percent']
scatter_matrix(train_set[scatter_features], figsize=(15, 15), alpha=0.3)
plt.suptitle('Scatter Matrix of Key Features', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

# Feature Transformations

## Transformations for MeanScore

In [ ]:
# Apply transformations to MeanScore
feature1 = 'MeanScore'
train_set[f'{feature1}_squared'] = train_set[feature1] ** 2
train_set[f'{feature1}_cubed'] = train_set[feature1] ** 3
train_set[f'{feature1}_log'] = np.log(train_set[feature1])
train_set[f'{feature1}_exp'] = np.exp(train_set[feature1] / 100)  # Scaled down to avoid overflow

# Plot histograms for MeanScore transformations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Transformations of {feature1}', fontsize=16)

axes[0, 0].hist(train_set[feature1], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title(f'Original {feature1}')

axes[0, 1].hist(train_set[f'{feature1}_squared'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title(f'{feature1} Squared')

axes[0, 2].hist(train_set[f'{feature1}_cubed'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 2].set_title(f'{feature1} Cubed')

axes[1, 0].hist(train_set[f'{feature1}_log'], bins=50, edgecolor='black', alpha=0.7, color='red')
axes[1, 0].set_title(f'{feature1} Log')

axes[1, 1].hist(train_set[f'{feature1}_exp'], bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].set_title(f'{feature1} Exponential (scaled)')

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## Transformations for Teacher_Count

In [ ]:
# Apply transformations to Teacher_Count
feature2 = 'Teacher_Count'
train_set[f'{feature2}_squared'] = train_set[feature2] ** 2
train_set[f'{feature2}_cubed'] = train_set[feature2] ** 3
train_set[f'{feature2}_log'] = np.log(train_set[feature2] + 1)
train_set[f'{feature2}_exp'] = np.exp(train_set[feature2] / 100)  # Scaled down to avoid overflow

# Plot histograms for Teacher_Count transformations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Transformations of {feature2}', fontsize=16)

axes[0, 0].hist(train_set[feature2], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title(f'Original {feature2}')

axes[0, 1].hist(train_set[f'{feature2}_squared'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title(f'{feature2} Squared')

axes[0, 2].hist(train_set[f'{feature2}_cubed'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 2].set_title(f'{feature2} Cubed')

axes[1, 0].hist(train_set[f'{feature2}_log'], bins=50, edgecolor='black', alpha=0.7, color='red')
axes[1, 0].set_title(f'{feature2} Log')

axes[1, 1].hist(train_set[f'{feature2}_exp'], bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].set_title(f'{feature2} Exponential (scaled)')

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## Scatter Matrices with Transformations

In [ ]:
# Scatter matrix for original features
original_features = ['MeanScore', 'Teacher_Count']
scatter_matrix(train_set[original_features], figsize=(10, 10), alpha=0.3)
plt.suptitle('Scatter Matrix: Original Features', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter matrix for squared transformations
squared_features = ['MeanScore_squared', 'Teacher_Count_squared']
scatter_matrix(train_set[squared_features], figsize=(10, 10), alpha=0.3)
plt.suptitle('Scatter Matrix: Squared Transformations', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter matrix for cubed transformations
cubed_features = ['MeanScore_cubed', 'Teacher_Count_cubed']
scatter_matrix(train_set[cubed_features], figsize=(10, 10), alpha=0.3)
plt.suptitle('Scatter Matrix: Cubed Transformations', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

# Save Cleaned Datasets

In [ ]:
# Save cleaned training and test sets to CSV
train_set.to_csv('/Users/vikobal/Regents_Data/machinelearning/schools_train.csv', index=False)
test_set.to_csv('/Users/vikobal/Regents_Data/machinelearning/schools_test.csv', index=False)

print("✓ Cleaned datasets saved successfully!")
print(f"  - schools_train.csv ({train_set.shape})")
print(f"  - schools_test.csv ({test_set.shape})")

# Summary and Conclusions

## Dataset Summary:
- **Original size:** 1,183 rows, 40 columns
- **Target variable:** Subject (3 classes: common_core_algebra, common_core_english, living_environment)
- **Features:** School demographics, staff statistics, student demographics, test scores
- **Training set:** 946 rows
- **Test set:** 237 rows

## Key Findings:
1. **Missing Values:** Handled using median imputation for numerical features
2. **Class Balance:** Dataset is balanced across the three subjects (equal distribution)
3. **Feature Distributions:**
   - MeanScore: Approximately normal distribution centered around 65-70
   - Teacher_Count: Right-skewed distribution with most schools having 40-60 teachers
   - Demographics show high variability across schools

4. **Transformation Effects:**
   - Log transformation helps normalize skewed distributions
   - Squared and cubed transformations amplify differences between high and low values
   - These transformations may be useful for modeling non-linear relationships